# Create AnnData from reference DRG Studies

- Sharma et al., Nature, 2020: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE139088
- Qi et al., Cell, 2024: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE254789

Before running this notebook, download the reference raw data using the following commands to download to your DATA_DIR (specified in config file):

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE139088
- wget -O GSE139088_RAW.tar 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE139088&format=file'
- mkdir -p GSE139088
- tar -xvf GSE139088_RAW.tar -C GSE139088

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE254789
- wget -O GSE254789_RAW.tar 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE254789&format=file'
- mkdir -p GSE254789
- tar -xvf GSE254789_RAW.tar -C GSE254789

This script creates the raw h5ad files from processed GEO uploads, maintaining appropriate metadata and organization.

**Pinned Environment:** [`envs/sc-scvi.yaml`](../../envs/sc-scvi.yaml)  

In [ ]:
import time
from pathlib import Path
import pandas as pd
import scanpy as sc
import anndata as ad
import re
import sys
import session_info
import os

### Directories

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[1]))

from config.paths import BASE_DIR
from config.paths import DATA_DIR
print('BASE_DIR:', BASE_DIR)
print('DATA_DIR:', DATA_DIR)

dataset1_dir = DATA_DIR / "GSE139088" #2019
dataset2_dir = DATA_DIR / "GSE254789" #2024, multiple samples

output_dir = BASE_DIR / "data" / "scrna-seq" / "h5ad" / "01_raw"
output_dir.mkdir(parents=True, exist_ok=True)
print('output_dir:', output_dir)

### Files

In [ ]:
dataset1_input = dataset1_dir / "GSM4130750_WT_1.csv.gz"
dataset2_input = sorted(dataset2_dir.glob("*.h5"))

print("dataset2_input files:")
for path in dataset2_input:
    print(f"- {path}")

In [ ]:
dataset1_input

# Create AnnData from the studies

## Dataset 1 - GSE139088
This is a 2019 study, all of the cells are in this processed file: `GSM4130750_WT_1.csv`.

11,140 cells.

In [ ]:
df1 = pd.read_csv(dataset1_input, low_memory=False)

In [ ]:
# Extract cell barcodes and annotations from first row
cell_barcodes = df1.iloc[0, 1:].values  # first row (barcodes), skip first column

# Column names are the original annotations but forced unique by pandas (.1, .2, etc.) so clean them:
cell_annotations = df1.columns[1:]
cell_annotations_clean = [re.sub(r'\.\d+$', '', ann) for ann in cell_annotations]

# Extract gene names and expression data
gene_names = df1.iloc[2:, 0].values
expression_data = df1.iloc[2:, 1:].astype(float).values

# Build AnnData
adata1 = ad.AnnData(
    X=expression_data.T,
    obs=pd.DataFrame(index=cell_barcodes, data={'original_annotation': cell_annotations_clean}),
    var=pd.DataFrame(index=gene_names)
)

adata1.var_names_make_unique()

# Add GSE + GSM metadata
adata1.obs['GSE'] = 'GSE139088'
adata1.obs['GSM'] = 'GSM4130750'

# Extract sample_number from barcode suffix
adata1.obs['sample_number'] = adata1.obs.index.str.split('-').str[1].astype(int)

# Reformat obs_names to "GSM4130750_<barcode>"
adata1.obs_names = [f"GSM4130750_{barcode}" for barcode in adata1.obs.index]

## Dataset 2 - GSE254789
This is a 2024 study with 92,895 cells unfiltered

In [ ]:
adatas2 = []

for path in dataset2_input:
    # Extract GSM ID and Sample number from filename
    stem_parts = path.stem.split("_")  # e.g. ['GSM8057824', 'Sample1', 'filtered', ...]
    gsm_id = stem_parts[0]  # 'GSM8057824'
    
    # Extract integer from 'Sample1' -> '1'
    sample_number_raw = stem_parts[1]  # 'Sample1'
    sample_number = sample_number_raw.lower().replace('sample', '')  # just '1'
    
    # Read 10X h5 file
    adata = sc.read_10x_h5(path)
    
    # Make var_names unique to avoid concat errors
    adata.var_names_make_unique()
    
    # Annotate
    adata.obs['GSE'] = 'GSE254789'
    adata.obs['GSM'] = gsm_id
    adata.obs['sample_number'] = str(int(sample_number))  # force string
    
    # Make obs_names unique using GSM ID only
    adata.obs_names = [f"{gsm_id}_{cell}" for cell in adata.obs_names]
    
    adatas2.append(adata)

# Combine all Dataset 2 samples only
adata2 = ad.concat(
    adatas2,
    join='outer',
    keys=[adata.obs['GSM'].unique()[0] for adata in adatas2]
)

In [ ]:
# Add sample_id column
adata1.obs['sample_id'] = adata1.obs['GSE'].astype(str) + "_" + adata1.obs['sample_number'].astype(str)
adata2.obs['sample_id'] = adata2.obs['GSE'].astype(str) + "_" + adata2.obs['sample_number'].astype(str)

adata1.obs

# Concatenate data objects

In [ ]:
adata1.obs['experiment_id'] = 'GSE139088'
adata2.obs['experiment_id'] = 'GSE254789'

# Now concatenate
adata_combined = ad.concat(
    [adata1, adata2],
    join='outer',
    label='experiment_id',   # this will add a 'experiment_id' column and preserve it
    keys=['GSE139088', 'GSE254789']
)


In [ ]:
adata_combined.obs['sample_number'] = adata_combined.obs['sample_number'].astype(str)

# Export

In [ ]:
output_dir

In [ ]:
adata1.write(output_dir / "GSE139088.h5ad", compression='gzip')
adata2.write(output_dir / "GSE254789.h5ad", compression='gzip')
adata_combined.write(output_dir / "drg_combined.h5ad", compression='gzip')